# Phase 3: fine-tune on public data and measure skin-tone performance honestly

**Decision-support prototype only. Not a diagnosis. Consult a qualified clinician.**

What this notebook does, in order:
1. Starts from the trained EfficientNet-B3 in the GitHub repo (no retraining from scratch).
2. Fine-tunes it on **HAM10000 + PAD-UFES-20** (2,298 smartphone photos, CC BY), so it sees real phone photos, which is how the app would be used.
3. Measures the original and the fine-tuned model on the **same held-out sets**, with 95% confidence intervals:
   - HAM10000 test (the original test set, frozen)
   - PAD-UFES-20 test (split by patient, never trained on)
   - **Dark skin, benign:** ISIC Archive images labelled Fitzpatrick V-VI from Memorial Sloan Kettering (one image per lesion). These have no specific diagnosis, only "benign", so they measure **false alarms** on dark skin, not cancer detection.
   - **Dark skin, labelled:** Fitzpatrick17k images labelled Fitzpatrick V-VI that map to our 7 classes and still download (small, so results are indicative only).
4. Applies a **pre-registered promotion rule**: the fine-tuned model is only exported for the app if it does not get worse on the original HAM10000 test set and does not raise false alarms on dark skin.

**Honest scope.** Public data with dark skin and cancer labels is extremely scarce. This notebook can show whether the model got better on phone photos and whether it raises false alarms on dark skin. It can NOT show that the model detects cancer well on dark skin. Nothing here allows that claim.

Runtime: *Runtime > Change runtime type > GPU (T4)*. Everything is saved to Google Drive; if Colab disconnects, run all cells again and training resumes.

## 1. Setup

In [ ]:
import os, sys, io, json, time, random, subprocess, urllib.request, urllib.parse
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from PIL import Image, ImageFile
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, precision_recall_fscore_support, roc_auc_score
ImageFile.LOAD_TRUNCATED_IMAGES = True

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("PHASE3_SMOKE") == "1"      # tiny local test mode (not used in Colab)

# ---------------- ONE CONFIG BLOCK: every number lives here ----------------
CFG = dict(
    repo_url="https://github.com/Elvis-Kayonga/ai-skin-lesion-detection-system-capstone-project.git",
    seed=42,
    class_codes=["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"],
    high_risk=["akiec", "bcc", "mel"],          # pre-cancer / cancer classes
    input_size=300, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225],
    pad_share=0.4,          # share of each training epoch drawn from PAD-UFES-20 (rest from HAM10000)
    pad_folds=7,            # same 7-fold recipe as the original split: fold 0 test, fold 1 val, rest train
    epochs=8, patience=3, batch_size=32,
    lr_backbone=1e-4, lr_head=3e-4, weight_decay=1e-4,   # small learning rates: adapt, do not forget
    mc_passes=10, mc_seed=1234,
    entropy_percentile=80.0, mel_recall_target=0.90,     # same threshold recipe as the training notebook
    bootstrap=1000,
    promote=dict(ham_macro_f1_drop=0.01, ham_mel_sens_drop=0.03, dark_false_alarm_rise=0.02),
    max_per_source=None,    # None = use all images
    workers=16,
    num_workers=2 if "google.colab" in sys.modules else 0,   # DataLoader worker processes (0 = safe on Windows)
)
if SMOKE:
    CFG.update(epochs=1, patience=1, batch_size=4, bootstrap=50, max_per_source=40, workers=8, mc_passes=3, pad_folds=3)

random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CODES = CFG["class_codes"]; IDX = {c: i for i, c in enumerate(CODES)}
MEL = IDX["mel"]; HR_IDX = [IDX[c] for c in CFG["high_risk"]]

if IN_COLAB:
    from google.colab import drive, files
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/skin_capstone/phase3"     # results + checkpoints survive disconnects
    WORK = "/content/work"                                    # images (fast local disk, re-downloaded if lost)
else:
    OUT = os.environ.get("PHASE3_OUT", "phase3_out"); WORK = os.environ.get("PHASE3_WORK", "phase3_work")
os.makedirs(OUT, exist_ok=True); os.makedirs(WORK, exist_ok=True)

# The trained model and the frozen HAM10000 split come from the public GitHub repo
REPO = os.environ.get("PHASE3_REPO") or f"{WORK}/repo"
if not os.path.exists(f"{REPO}/backend/model/efficientnet_b3_best.pth"):
    subprocess.run(["git", "clone", "--depth", "1", CFG["repo_url"], REPO], check=True)
ORIG_CFG = json.load(open(f"{REPO}/backend/model/model_config.json"))
print("device:", device, "| smoke mode:", SMOKE, "| outputs:", OUT)

In [ ]:
# Small download helpers (retries, browser-like user agent, image check before saving)
UA = {"User-Agent": "Mozilla/5.0 (capstone research; non-commercial)"}

def http_json(url):
    for attempt in range(4):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=60) as r:
                return json.load(r)
        except Exception:
            if attempt == 3: raise
            time.sleep(2 * (attempt + 1))

def isic_search(params):
    """All results of an ISIC Archive image search (public API, no login)."""
    url = "https://api.isic-archive.com/api/v2/images/search/?" + urllib.parse.urlencode({**params, "limit": 100})
    out = []
    while url:
        page = http_json(url); out += page["results"]; url = page.get("next")
    return out

def download_one(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return True
    for attempt in range(3):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=30) as r:
                data = r.read()
            Image.open(io.BytesIO(data)).verify()              # reject HTML error pages / broken files
            with open(path, "wb") as f: f.write(data)
            return True
        except Exception:
            time.sleep(1 + attempt)
    return False

def download_many(df):
    if len(df) == 0:
        return df
    os.makedirs(os.path.dirname(df["path"].iloc[0]), exist_ok=True)
    with ThreadPoolExecutor(CFG["workers"]) as ex:
        ok = list(ex.map(lambda a: download_one(*a), zip(df["url"], df["path"])))
    print(f"   downloaded {sum(ok)} of {len(ok)}")
    return df[np.array(ok, dtype=bool)].reset_index(drop=True)

def limit(df):
    n = CFG["max_per_source"]
    return df if not n or len(df) <= n else df.sample(n, random_state=CFG["seed"]).reset_index(drop=True)

clin = lambda r, k: (r.get("metadata", {}).get("clinical") or {}).get(k)
ext_of = lambda url: os.path.splitext(urllib.parse.urlparse(url).path)[1] or ".jpg"

## 2. Data

| Source | Use | Licence |
|---|---|---|
| HAM10000 (Kaggle) | train / val / test, original frozen split | CC BY-NC 4.0 |
| PAD-UFES-20 via ISIC Archive collection 406 | train / val / test, split by patient | CC BY |
| ISIC Archive, Fitzpatrick V-VI, Memorial Sloan Kettering | **test only** (benign false alarms) | CC BY |
| Fitzpatrick17k, Fitzpatrick V-VI | **test only** | CC BY-NC-SA 3.0 |

Squamous cell carcinoma (SCC) images are dropped because HAM10000 has no SCC class. Images are never committed to the repo.

In [ ]:
# ---- PAD-UFES-20 (smartphone photos) via the ISIC Archive ----
PAD_MAP = {"Basal cell carcinoma": "bcc", "Solar or actinic keratosis": "akiec", "Nevus": "nv",
           "Seborrheic keratosis": "bkl", "Melanoma, NOS": "mel"}          # SCC deliberately dropped
rows = []
for r in isic_search({"collections": 406}):
    d3 = clin(r, "diagnosis_3")
    if d3 in PAD_MAP:
        url = r["files"]["full"]["url"]
        rows.append(dict(image_id=r["isic_id"], url=url, path=f"{WORK}/pad/{r['isic_id']}{ext_of(url)}",
                         dx=PAD_MAP[d3], group=clin(r, "patient_id"), fitz=clin(r, "fitzpatrick_skin_type"),
                         source="PAD-UFES-20"))
pad = download_many(limit(pd.DataFrame(rows)))

# Split by PATIENT so no patient appears in two sets (same 7-fold recipe as the original split)
sgkf = StratifiedGroupKFold(n_splits=CFG["pad_folds"], shuffle=True, random_state=CFG["seed"])
folds = [te for _, te in sgkf.split(pad, pad["dx"], groups=pad["group"])]
pad_test, pad_val = pad.iloc[folds[0]].reset_index(drop=True), pad.iloc[folds[1]].reset_index(drop=True)
pad_train = pad.iloc[np.concatenate(folds[2:])].reset_index(drop=True)
assert not set(pad_train.group) & set(pad_test.group) and not set(pad_train.group) & set(pad_val.group)
print("PAD-UFES-20:", {k: len(v) for k, v in [("train", pad_train), ("val", pad_val), ("test", pad_test)]})
print("PAD Fitzpatrick types:", pad["fitz"].value_counts(dropna=False).to_dict())

In [ ]:
# ---- HAM10000 with the ORIGINAL frozen split (from the repo) ----
splits = {n: pd.read_csv(f"{REPO}/docs/results/split_{n}.csv") for n in ["train", "val", "test"]}
if IN_COLAB:
    import glob
    if not os.path.exists("/root/.kaggle/kaggle.json"):
        up = files.upload()   # choose kaggle.json (stored only in this Colab session)
        os.makedirs("/root/.kaggle", exist_ok=True)
        open("/root/.kaggle/kaggle.json", "wb").write(up["kaggle.json"]); os.chmod("/root/.kaggle/kaggle.json", 0o600)
    if not glob.glob(f"{WORK}/ham/**/*.jpg", recursive=True):
        subprocess.run(["kaggle", "datasets", "download", "-d", "kmader/skin-cancer-mnist-ham10000",
                        "-p", f"{WORK}/ham", "--unzip", "-q"], check=True)
    index = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(f"{WORK}/ham/**/*.jpg", recursive=True)}
    def ham(name):
        d = splits[name].copy(); d["path"] = d["image_id"].map(index)
        assert d["path"].notna().all(), "some HAM10000 images are missing"
        d["group"] = d["lesion_id"]; d["source"] = "HAM10000"; d["fitz"] = None
        return limit(d)
    ham_train, ham_val, ham_test = ham("train"), ham("val"), ham("test")
    orig_counts = pd.read_csv(f"{REPO}/docs/results/split_class_counts.csv").set_index("dx")
    same = all(int(splits[n]["dx"].value_counts().get(c, 0)) == int(orig_counts.loc[c, n]) for n in splits for c in orig_counts.index)
    print("HAM10000 split identical to the original run:", same)
    assert same, "HAM10000 split does not match the original run: stop and check docs/results/split_*.csv"
else:
    # Local smoke test only: HAM10000 is not on this machine, so PAD images stand in for it.
    print("LOCAL SMOKE MODE: using PAD-UFES-20 images as a stand-in for HAM10000 (results meaningless).")
    ham_train, ham_val, ham_test = (d.assign(source="HAM10000 (smoke stand-in)") for d in (pad_train, pad_val, pad_test))
print("HAM10000:", {k: len(v) for k, v in [("train", ham_train), ("val", ham_val), ("test", ham_test)]})

In [ ]:
# ---- Dark skin, benign (TEST ONLY): ISIC Archive images labelled Fitzpatrick V-VI ----
dark_recs = isic_search({"query": "fitzpatrick_skin_type:V OR fitzpatrick_skin_type:VI"})
rows = []
for r in sorted(dark_recs, key=lambda r: r["isic_id"]):
    if (r.get("attribution") or "").startswith("Memorial Sloan Kettering") and clin(r, "diagnosis_1") == "Benign":
        url = r["files"]["full"]["url"]
        rows.append(dict(image_id=r["isic_id"], url=url, path=f"{WORK}/dark_benign/{r['isic_id']}{ext_of(url)}",
                         dx=None, lesion=clin(r, "lesion_id"), group=clin(r, "patient_id"),
                         fitz=clin(r, "fitzpatrick_skin_type"),
                         image_type=(r["metadata"].get("acquisition") or {}).get("image_type"),
                         source="ISIC Fitzpatrick V-VI benign (MSK)"))
dark_benign = pd.DataFrame(rows).drop_duplicates("lesion")      # one image per lesion
dark_benign = download_many(limit(dark_benign))
print("dark-skin benign test set:", len(dark_benign), "images |", dark_benign["group"].nunique(), "patients |",
      dark_benign["image_type"].value_counts().to_dict())

# ---- Dark skin, labelled (TEST ONLY): Fitzpatrick17k, Fitzpatrick V-VI, labels that map to our classes ----
FITZ_MAP = {"melanoma": "mel", "superficial spreading melanoma ssm": "mel", "malignant melanoma": "mel",
            "lentigo maligna": "mel", "basal cell carcinoma": "bcc", "solid cystic basal cell carcinoma": "bcc",
            "basal cell carcinoma morpheiform": "bcc", "actinic keratosis": "akiec", "seborrheic keratosis": "bkl",
            "nevocytic nevus": "nv", "halo nevus": "nv", "congenital nevus": "nv", "dermatofibroma": "df",
            "pyogenic granuloma": "vasc", "granuloma pyogenic": "vasc"}
fz = pd.read_csv("https://raw.githubusercontent.com/mattgroh/fitzpatrick17k/main/fitzpatrick17k.csv")
fz = fz[fz["label"].isin(FITZ_MAP) & fz["fitzpatrick_scale"].isin([5, 6])].drop_duplicates("md5hash")
dark_fitz = pd.DataFrame(dict(image_id=fz["md5hash"], url=fz["url"], path=[f"{WORK}/fitz/{h}.jpg" for h in fz["md5hash"]],
                              dx=fz["label"].map(FITZ_MAP), group=fz["md5hash"], fitz=fz["fitzpatrick_scale"],
                              source="Fitzpatrick17k V-VI"))
print("Fitzpatrick17k V-VI candidates:", len(dark_fitz), "(many source links are broken)")
dark_fitz = download_many(limit(dark_fitz))
print("Fitzpatrick17k V-VI usable:", len(dark_fitz), dark_fitz["dx"].value_counts().to_dict())

In [ ]:
# ---- Data manifest (saved with the results) ----
for d in (ham_train, ham_val, ham_test, pad_train, pad_val, pad_test, dark_fitz):
    d["label"] = d["dx"].map(IDX).astype(int)
dark_benign["label"] = -1
manifest = {name: dict(n=int(len(d)), classes=d["dx"].value_counts().to_dict() if d["dx"].notna().any() else "benign (no class)")
            for name, d in [("ham_train", ham_train), ("ham_val", ham_val), ("ham_test", ham_test),
                            ("pad_train", pad_train), ("pad_val", pad_val), ("pad_test", pad_test),
                            ("dark_benign_test", dark_benign), ("dark_fitz17k_test", dark_fitz)]}
manifest["licences"] = {"HAM10000": "CC BY-NC 4.0", "PAD-UFES-20": "CC BY (Pacheco et al. 2020, ISIC collection 406)",
                        "ISIC Fitzpatrick V-VI (MSK)": "CC BY (Memorial Sloan Kettering Cancer Center)",
                        "Fitzpatrick17k": "CC BY-NC-SA 3.0 (Groh et al. 2021)"}
json.dump(manifest, open(f"{OUT}/phase3_data_manifest.json", "w"), indent=2, default=str)
pd.concat([pad_train.assign(split="train"), pad_val.assign(split="val"), pad_test.assign(split="test")])[
    ["image_id", "group", "dx", "fitz", "split"]].to_csv(f"{OUT}/phase3_pad_split.csv", index=False)
print(json.dumps(manifest, indent=1, default=str))

## 3. Model, inference and metrics
Same inference contract as the app: resize straight to 300x300, ImageNet normalisation, backbone once, classifier run 10 times with dropout on (MC Dropout), average the softmax.

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((CFG["input_size"], CFG["input_size"])),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.03),     # lighting / camera variation
    transforms.RandomApply([transforms.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.2),        # phone-photo softness
    transforms.ToTensor(), transforms.Normalize(CFG["mean"], CFG["std"])])
eval_tf = transforms.Compose([transforms.Resize((CFG["input_size"], CFG["input_size"])),
                              transforms.ToTensor(), transforms.Normalize(CFG["mean"], CFG["std"])])

class ImgSet(Dataset):
    def __init__(self, df, tf): self.paths, self.labels, self.tf = df["path"].values, df["label"].values, tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return self.tf(Image.open(self.paths[i]).convert("RGB")), int(self.labels[i])

def build_model():
    m = models.efficientnet_b3(weights=None)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, len(CODES))
    return m

def load_original():
    m = build_model()
    m.load_state_dict(torch.load(f"{REPO}/backend/model/efficientnet_b3_best.pth", map_location="cpu"))
    return m.to(device).eval()

@torch.no_grad()
def predict(model, df):
    """Returns (MC Dropout averaged probabilities, plain eval-mode probabilities)."""
    torch.manual_seed(CFG["mc_seed"])
    P_mc, P_plain = [], []
    model.eval()
    for x, _ in DataLoader(ImgSet(df, eval_tf), batch_size=32, shuffle=False, num_workers=CFG["num_workers"]):
        feats = F.adaptive_avg_pool2d(model.features(x.to(device)), 1).flatten(1)
        P_plain.append(F.softmax(model.classifier(feats), 1).cpu())
        model.classifier.train()                                   # dropout ON (classifier only)
        P_mc.append(torch.stack([F.softmax(model.classifier(feats), 1) for _ in range(CFG["mc_passes"])]).mean(0).cpu())
        model.classifier.eval()
    return torch.cat(P_mc).numpy(), torch.cat(P_plain).numpy()

entropy = lambda P: -(P * np.log(P + 1e-12)).sum(1)

def thresholds_from_val(P_val, y_val):
    """Same recipe as the training notebook, chosen on VALIDATION only."""
    ent = float(np.percentile(entropy(P_val), CFG["entropy_percentile"]))
    is_mel = y_val == MEL
    ok = [t for t in np.linspace(0.05, 0.95, 91) if is_mel.any() and (P_val[is_mel, MEL] >= t).mean() >= CFG["mel_recall_target"]]
    return ent, float(max(ok)) if ok else 0.05

def cluster_ci(fn, groups, n_boot):
    """95% CI by resampling whole groups (patients or lesions) so near-duplicate photos do not fake precision."""
    groups = np.asarray(groups); uniq = np.unique(groups); pos = {g: np.where(groups == g)[0] for g in uniq}
    rng = np.random.default_rng(CFG["seed"]); vals = []
    for _ in range(n_boot):
        idx = np.concatenate([pos[g] for g in rng.choice(uniq, len(uniq), replace=True)])
        v = fn(idx)
        if v is not None and np.isfinite(v): vals.append(v)
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))) if vals else (np.nan, np.nan)

def evaluate(P_mc, P_plain, df, ent_thr, mel_thr):
    """Metrics with 95% CIs. Labelled sets get accuracy/F1/sensitivity; the benign dark-skin set gets false-alarm rates."""
    pred = P_plain.argmax(1); flag = (entropy(P_mc) >= ent_thr) | (P_mc[:, MEL] >= mel_thr)
    y = df["label"].values; g = df["group"].values; out = {"n": int(len(df)), "patients_or_groups": int(len(np.unique(g)))}
    def add(name, fn):
        v = fn(np.arange(len(y))); lo, hi = cluster_ci(fn, g, CFG["bootstrap"])
        out[name] = None if v is None else round(float(v), 4); out[name + "_ci95"] = [round(lo, 4), round(hi, 4)]
    if (y >= 0).all():
        labels = np.unique(y); hr_true = np.isin(y, HR_IDX)
        add("accuracy", lambda i: float((pred[i] == y[i]).mean()))
        add("macro_f1", lambda i: float(f1_score(y[i], pred[i], labels=labels, average="macro", zero_division=0)))
        add("melanoma_sensitivity", lambda i: float((pred[i][y[i] == MEL] == MEL).mean()) if (y[i] == MEL).any() else None)
        add("high_risk_sensitivity", lambda i: float(np.isin(pred[i][hr_true[i]], HR_IDX).mean()) if hr_true[i].any() else None)
        add("high_risk_sensitivity_with_flag",
            lambda i: float((np.isin(pred[i], HR_IDX) | flag[i])[hr_true[i]].mean()) if hr_true[i].any() else None)
        add("review_flag_rate", lambda i: float(flag[i].mean()))
    else:   # benign only: every high-risk call and every melanoma call is a false alarm
        add("false_alarm_high_risk_rate", lambda i: float(np.isin(pred[i], HR_IDX).mean()))
        add("false_alarm_melanoma_rate", lambda i: float((pred[i] == MEL).mean()))
        add("review_flag_rate", lambda i: float(flag[i].mean()))
    return out

EVAL_SETS = [("HAM10000 test", ham_test), ("PAD-UFES-20 test (phone photos)", pad_test),
             ("Dark skin V-VI benign (ISIC/MSK)", dark_benign), ("Dark skin V-VI labelled (Fitzpatrick17k)", dark_fitz)]
EVAL_SETS = [(n, d) for n, d in EVAL_SETS if len(d)]

## 4. Baseline: the ORIGINAL model on every evaluation set

In [ ]:
orig = load_original()
ORIG_THR = (ORIG_CFG["entropy_threshold"], ORIG_CFG["melanoma_flag_threshold"])   # what the app uses today
results = []
for name, d in EVAL_SETS:
    P_mc, P_plain = predict(orig, d)
    r = evaluate(P_mc, P_plain, d, *ORIG_THR); r.update(model="original", eval_set=name); results.append(r)
    print(name, {k: v for k, v in r.items() if not k.endswith("_ci95")})

## 5. Fine-tune (resumable)
Starts from the original weights with small learning rates. Each epoch mixes HAM10000 (60%) and PAD-UFES-20 (40%) so the model learns phone photos without forgetting dermoscopy. Early stopping uses the average of HAM10000-val and PAD-val macro-F1. A checkpoint is written to Google Drive after every epoch.

In [ ]:
def save_atomic(obj, path):
    torch.save(obj, path + ".tmp"); os.replace(path + ".tmp", path)

def val_macro_f1(model, df):
    _, P = predict(model, df)
    return float(f1_score(df["label"].values, P.argmax(1), labels=np.unique(df["label"].values), average="macro", zero_division=0))

train_df = pd.concat([ham_train, pad_train], ignore_index=True)
is_pad = (train_df["source"] == "PAD-UFES-20").values
w = np.where(is_pad, CFG["pad_share"] / max(is_pad.sum(), 1), (1 - CFG["pad_share"]) / max((~is_pad).sum(), 1))
counts = np.bincount(train_df["label"].values, minlength=len(CODES)).astype(float); counts[counts == 0] = 1
class_w = torch.tensor(counts.sum() / (len(CODES) * counts), dtype=torch.float32)

def finetune():
    model = load_original().train()
    crit = nn.CrossEntropyLoss(weight=class_w.to(device))
    opt = torch.optim.AdamW([{"params": model.features.parameters(), "lr": CFG["lr_backbone"]},
                             {"params": model.classifier.parameters(), "lr": CFG["lr_head"]}], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["epochs"])
    scaler = torch.amp.GradScaler(device.type, enabled=device.type == "cuda")
    ck, best_path = f"{OUT}/finetune_checkpoint.pt", f"{OUT}/finetuned_best.pth"
    best, bad, hist, start, done = -1.0, 0, [], 0, False
    if os.path.exists(ck):                                         # ---- resume ----
        c = torch.load(ck, map_location=device, weights_only=False)
        model.load_state_dict(c["model"]); opt.load_state_dict(c["opt"]); sched.load_state_dict(c["sched"]); scaler.load_state_dict(c["scaler"])
        best, bad, hist, start, done = c["best"], c["bad"], c["hist"], c["epoch"], c["done"]
        print(f"resumed: {start} epochs done, best {best:.4f}" + (" (finished)" if done else ""))
    def checkpoint(epoch, finished):
        save_atomic(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), scaler=scaler.state_dict(),
                         best=best, bad=bad, hist=hist, epoch=epoch, done=finished), ck)
    loader = DataLoader(ImgSet(train_df, train_tf), batch_size=CFG["batch_size"], num_workers=CFG["num_workers"], drop_last=True,
                        sampler=WeightedRandomSampler(torch.tensor(w, dtype=torch.double), num_samples=len(train_df), replacement=True))
    for ep in range(start, 0 if done else CFG["epochs"]):
        model.train(); t0, tot, n = time.time(), 0.0, 0
        for x, y in loader:
            x, y = x.to(device), y.to(device); opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); tot += float(loss) * len(y); n += len(y)
        sched.step()
        f_ham, f_pad = val_macro_f1(model, ham_val), val_macro_f1(model, pad_val); score = (f_ham + f_pad) / 2
        hist.append(dict(epoch=ep + 1, train_loss=tot / max(n, 1), val_macro_f1_ham=f_ham, val_macro_f1_pad=f_pad, val_score=score))
        print(f"ep {ep+1}/{CFG['epochs']} loss {tot/max(n,1):.4f} | val macro-F1 HAM {f_ham:.4f} PAD {f_pad:.4f} | {time.time()-t0:.0f}s")
        if score > best:
            best, bad = score, 0; save_atomic(model.state_dict(), best_path)
        else:
            bad += 1
        stop = bad >= CFG["patience"]
        checkpoint(ep + 1, finished=stop or ep + 1 == CFG["epochs"])
        if stop: print("early stop"); break
    if not done: checkpoint(len(hist), finished=True)
    model.load_state_dict(torch.load(best_path, map_location=device))
    pd.DataFrame(hist).to_csv(f"{OUT}/phase3_finetune_history.csv", index=False)
    return model.eval(), hist

ft, ft_hist = finetune()

## 6. Fine-tuned model: new thresholds (validation only) and the same evaluation

In [ ]:
val_df = pd.concat([ham_val, pad_val], ignore_index=True)
P_val, _ = predict(ft, val_df)
FT_THR = thresholds_from_val(P_val, val_df["label"].values)
print(f"fine-tuned thresholds (validation): entropy {FT_THR[0]:.4f}, melanoma {FT_THR[1]:.2f}")
for name, d in EVAL_SETS:
    P_mc, P_plain = predict(ft, d)
    r = evaluate(P_mc, P_plain, d, *FT_THR); r.update(model="fine-tuned", eval_set=name); results.append(r)

res = pd.DataFrame(results)
res.to_csv(f"{OUT}/phase3_eval.csv", index=False)
show = [c for c in ["accuracy", "macro_f1", "melanoma_sensitivity", "high_risk_sensitivity", "high_risk_sensitivity_with_flag",
                    "false_alarm_high_risk_rate", "false_alarm_melanoma_rate", "review_flag_rate"] if c in res]
table = res.pivot_table(index="eval_set", columns="model", values=show, aggfunc="first")
pd.set_option("display.width", 220); print(table.round(3))

## 7. Pre-registered promotion rule
The fine-tuned model replaces the app's model ONLY if all of these hold (decided before seeing results):
1. HAM10000 test macro-F1 drops by no more than 0.01.
2. HAM10000 test melanoma sensitivity drops by no more than 0.03.
3. PAD-UFES-20 (phone photos) test macro-F1 improves.
4. False alarms (high-risk calls) on benign dark-skin lesions rise by no more than 0.02.

In [ ]:
def get(model, s, m):
    row = res[(res.model == model) & (res.eval_set == s)]
    return float(row[m].iloc[0]) if len(row) and m in row and pd.notna(row[m].iloc[0]) else np.nan
P = CFG["promote"]; H, PD_, DK = "HAM10000 test", "PAD-UFES-20 test (phone photos)", "Dark skin V-VI benign (ISIC/MSK)"
checks = {
    "HAM macro-F1 not worse":        get("fine-tuned", H, "macro_f1") >= get("original", H, "macro_f1") - P["ham_macro_f1_drop"],
    "HAM melanoma sens. not worse":  get("fine-tuned", H, "melanoma_sensitivity") >= get("original", H, "melanoma_sensitivity") - P["ham_mel_sens_drop"],
    "PAD macro-F1 improved":         get("fine-tuned", PD_, "macro_f1") > get("original", PD_, "macro_f1"),
    "dark-skin false alarms not up": (not np.isfinite(get("original", DK, "false_alarm_high_risk_rate"))) or
                                     get("fine-tuned", DK, "false_alarm_high_risk_rate") <= get("original", DK, "false_alarm_high_risk_rate") + P["dark_false_alarm_rise"],
}
PROMOTE = all(bool(v) for v in checks.values()) and not SMOKE
for k, v in checks.items(): print(("PASS " if v else "FAIL ") + k)
print("\nPROMOTE TO APP:", PROMOTE, "(smoke mode never promotes)" if SMOKE else "")

In [ ]:
# Summary + (if promoted) candidate model files with the SAME contract as backend/model/
def contract_metrics(model, d, thr):
    P_mc, P = predict(model, d); y = d["label"].values; pred = P.argmax(1)
    pr, rc, f1, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    try: auc = float(roc_auc_score(y, P, multi_class="ovr", average="macro", labels=list(range(len(CODES)))))
    except ValueError: auc = float("nan")
    flag = (entropy(P_mc) >= thr[0]) | (P_mc[:, MEL] >= thr[1]); mc_pred = P_mc.argmax(1); ok = mc_pred == y; mel = y == MEL
    test = dict(accuracy=float((pred == y).mean()), macro_precision=float(pr), macro_recall=float(rc), macro_f1=float(f1),
                macro_auc_ovr=auc, melanoma_sensitivity=float((pred[mel] == MEL).mean()) if mel.any() else None)
    mission = dict(mc_accuracy=float(ok.mean()), review_rate=float(flag.mean()),
                   accuracy_unflagged=float(ok[~flag].mean()) if (~flag).any() else None,
                   accuracy_flagged=float(ok[flag].mean()) if flag.any() else None,
                   errors_caught_by_flag=float(flag[~ok].mean()) if (~ok).any() else None,
                   melanoma_sensitivity_argmax=float(ok[mel].mean()) if mel.any() else None,
                   melanoma_sensitivity_with_flag=float((ok | flag)[mel].mean()) if mel.any() else None)
    return test, mission

summary = dict(promote=PROMOTE, checks={k: bool(v) for k, v in checks.items()}, finetuned_thresholds=FT_THR,
               original_thresholds=ORIG_THR, config={k: v for k, v in CFG.items() if k != "repo_url"}, history=ft_hist,
               honest_scope="Dark-skin results cover benign false alarms (few patients) and a small Fitzpatrick17k sample. "
                            "They do not show cancer-detection performance on Fitzpatrick V-VI.")
json.dump(summary, open(f"{OUT}/phase3_summary.json", "w"), indent=2, default=str)

test, mission = contract_metrics(ft, ham_test, FT_THR)
summary["finetuned_ham_test_metrics"], summary["finetuned_ham_mission_metrics"] = test, mission
json.dump(summary, open(f"{OUT}/phase3_summary.json", "w"), indent=2, default=str)
if PROMOTE:
    os.makedirs(f"{OUT}/model_candidate", exist_ok=True)
    cand = {k: v for k, v in ORIG_CFG.items() if k not in ("temperature", "calibrated_entropy_threshold",
                                                           "calibrated_melanoma_threshold", "calibration")}
    cand.update(model_name="efficientnet_b3_finetuned_ham_padufes", entropy_threshold=FT_THR[0],
                melanoma_flag_threshold=FT_THR[1], test_metrics=test, mission_metrics=mission,
                training_data="HAM10000 + PAD-UFES-20 (fine-tuned from the original EfficientNet-B3)",
                limitations="Fine-tuned on HAM10000 (dermoscopy) and PAD-UFES-20 (smartphone, mostly Fitzpatrick I-III). "
                            "Not validated for cancer detection on Fitzpatrick V-VI.")
    torch.save(ft.state_dict(), f"{OUT}/model_candidate/efficientnet_b3_best.pth")
    json.dump(cand, open(f"{OUT}/model_candidate/model_config.json", "w"), indent=2)
    print("candidate model written to", f"{OUT}/model_candidate (copy both files into backend/model/ only after review)")
else:
    print("No candidate exported: the app keeps the original model.")

In [ ]:
# One chart for the README / supervisor
import matplotlib.pyplot as plt
plot_rows = [(s, m) for s, m in [("HAM10000 test", "macro_f1"), ("PAD-UFES-20 test (phone photos)", "macro_f1"),
                                 ("Dark skin V-VI benign (ISIC/MSK)", "false_alarm_high_risk_rate"),
                                 ("Dark skin V-VI labelled (Fitzpatrick17k)", "accuracy")] if m in res]
fig, ax = plt.subplots(figsize=(10, 4)); xs = np.arange(len(plot_rows))
for k, (mdl, off) in enumerate([("original", -0.18), ("fine-tuned", 0.18)]):
    vals, los, his = [], [], []
    for s, m in plot_rows:
        row = res[(res.model == mdl) & (res.eval_set == s)]
        v = float(row[m].iloc[0]) if len(row) and pd.notna(row[m].iloc[0]) else np.nan
        lo, hi = row[m + "_ci95"].iloc[0] if len(row) else (np.nan, np.nan)
        vals.append(v); los.append(v - lo); his.append(hi - v)
    ax.bar(xs + off, vals, 0.36, yerr=[los, his], capsize=4, label=mdl)
ax.set_xticks(xs); ax.set_xticklabels([f"{s}\n{m}" for s, m in plot_rows], fontsize=8)
ax.set_ylim(0, 1); ax.legend(); ax.set_title("Original vs fine-tuned (95% CI). Lower is better for false alarms.")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_phase3_comparison.png", dpi=150)
if IN_COLAB: plt.show()

In [ ]:
# Download the results (checkpoints stay on Google Drive)
if IN_COLAB:
    import shutil
    keep = [f for f in os.listdir(OUT) if f.startswith(("phase3_", "fig_phase3")) or f == "model_candidate"]
    stage = "/content/phase3_results"; shutil.rmtree(stage, ignore_errors=True); os.makedirs(stage)
    for f in keep:
        src = f"{OUT}/{f}"
        (shutil.copytree if os.path.isdir(src) else shutil.copy)(src, f"{stage}/{f}")
    shutil.make_archive("/content/phase3_results", "zip", stage)
    files.download("/content/phase3_results.zip")
print("done")

### After the run
1. Unzip `phase3_results.zip` into the repo: `phase3_*.csv/json` and `fig_phase3_comparison.png` go to `docs/results/` and `docs/figures/`.
2. If `model_candidate/` exists (the promotion rule passed), review the numbers first, then copy its two files into `backend/model/` and restart the backend.
3. Report the results honestly, including any FAIL. A result where fine-tuning did not help is still a valid finding.